# Actividad 3 – Análisis de Datos: Liga de Naciones de Voleibol (VNL)
### Descripción:
Esta actividad se centra en aplicar y profundizar los conocimientos y habilidades que adquiriste en los temas 9 al 12. A través de un conjunto de datos reales, relacionados con el rendimiento de equipos de béisbol, tendrás la oportunidad de practicar técnicas esenciales en el proceso de ciencia de datos, desde la adquisición y preparación de datos hasta la modelación, predicción y evaluación de resultados.

### Objetivo:
Reforzar los conceptos de regresión lineal simple y limpieza de datos, utilizando datos reales de equipos de béisbol, para predecir el número de carreras (runs) basado en el número de bateos.

### Instrucciones:
Parte 1. Preparación de los datos

- Obtención de los datos: Guarda la base de datos en una variable, los datos los encontrarás en la siguiente página: https://www.espn.com.mx/beisbol/mlb/estadisticas/jugador

- Limpieza y preparación de los datos: Evalúa los datos recopilados en busca de valores faltantes o erróneos. Además, realiza la limpieza necesaria, la imputación de datos faltantes y la estandarización de los datos para asegurar su calidad y uniformidad.


---

## Parte 1: Preparación de los Datos

### 1 Obtención de los datos
Cargamos el dataset en una variable llamada df usando la librería pandas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Cargar el dataset
df = pd.read_csv("VNL.csv")

# Mostrar las primeras filas
print("Vista previa del dataset:")
df.head(10)

In [ ]:
# Dimensiones del dataset
print(f"El dataset tiene {df.shape[0]} filas y {df.shape[1]} columnas.")
print()
print("Columnas disponibles:")
for col in df.columns:
    print(f"  - {col}")

### 2 Limpieza y preparación de los datos

Revisamos si hay **valores faltantes, tipos de datos incorrectos o valores atípicos** que necesiten corrección.

In [ ]:
# Verificar valores nulos
print("=== Valores nulos por columna ===")
nulos = df.isnull().sum()
print(nulos)
print()
if nulos.sum() == 0:
    print("✅ No se encontraron valores nulos en el dataset.")
else:
    print("⚠️ Se encontraron valores nulos. Se procederá a imputarlos.")

In [ ]:
# Verificar tipos de datos
print("=== Tipos de datos ===")
print(df.dtypes)

In [ ]:
# Estadísticas descriptivas de las columnas numéricas
print("=== Estadísticas descriptivas ===")
df.describe().round(2)

In [ ]:
# Verificar que no haya valores negativos en columnas de estadísticas
columnas_numericas = ["Age", "Attack", "Block", "Serve", "Set", "Dig", "Receive"]
negativos = (df[columnas_numericas] < 0).sum()
print("=== Valores negativos por columna ===")
print(negativos)
print()
if negativos.sum() == 0:
    print("✅ No se encontraron valores negativos.")
else:
    print("⚠️ Se encontraron valores negativos.")

In [ ]:
# Verificar duplicados
duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")
if duplicados > 0:
    df = df.drop_duplicates()
    print(f"✅ Se eliminaron {duplicados} filas duplicadas.")
else:
    print("✅ No hay filas duplicadas.")

In [ ]:
# Imputación de valores faltantes (por si existieran en otros escenarios)
# Para columnas numéricas usamos la mediana; para categóricas la moda
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ["float64", "int64"]:
            mediana = df[col].median()
            df[col].fillna(mediana, inplace=True)
            print(f"Columna "{col}": imputada con mediana = {mediana}")
        else:
            moda = df[col].mode()[0]
            df[col].fillna(moda, inplace=True)
            print(f"Columna "{col}": imputada con moda = {moda}")

print("✅ Proceso de imputación completado.")

In [ ]:
# Estandarización de datos
# Creamos una copia estandarizada SOLO para visualización comparativa
# (el modelo usará los datos originales para que los resultados sean interpretables)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_estandarizado = df[columnas_numericas].copy()
df_estandarizado = pd.DataFrame(
    scaler.fit_transform(df_estandarizado),
    columns=columnas_numericas
)

print("=== Estadísticas tras estandarización (media≈0, desv.est≈1) ===")
df_estandarizado.describe().round(2)

In [ ]:
# Distribución de jugadores por posición
print("=== Jugadores por posición ===")
posiciones = {
    "OH": "Outside Hitter (Receptor/Atacante)",
    "OP": "Opposite (Opuesto)",
    "MB": "Middle Blocker (Central)",
    "S":  "Setter (Armador)",
    "L":  "Libero"
}
conteo = df["Position"].value_counts()
for pos, count in conteo.items():
    desc = posiciones.get(pos, pos)
    print(f"  {pos} – {desc}: {count} jugadores")

---
## Parte 2: Modelado y Evaluación

### 2.1 Análisis Exploratorio – Correlación de Pearson

Calculamos la correlación de Pearson entre **Edad (`Age`)** y **Puntos de Ataque (`Attack`)** para entender si existe una relación lineal entre estas dos variables.

In [ ]:
# Correlación de Pearson entre Age y Attack
correlacion, p_valor = stats.pearsonr(df["Age"], df["Attack"])

print("=== Correlación de Pearson: Age vs Attack ===")
print(f"  Coeficiente r = {correlacion:.4f}")
print(f"  Valor p       = {p_valor:.4f}")
print()

# Interpretación automática
abs_r = abs(correlacion)
if abs_r >= 0.7:
    fuerza = "fuerte"
elif abs_r >= 0.4:
    fuerza = "moderada"
else:
    fuerza = "débil"

direccion = "positiva" if correlacion > 0 else "negativa"

print(f"📊 Interpretación:")
print(f"  La correlación entre la edad y el ataque es {fuerza} y {direccion} (r = {correlacion:.4f}).")
if p_valor < 0.05:
    print(f"  El valor p ({p_valor:.4f}) < 0.05, por lo que la correlación ES estadísticamente significativa.")
else:
    print(f"  El valor p ({p_valor:.4f}) >= 0.05, por lo que la correlación NO es estadísticamente significativa.")

In [ ]:
# Matriz de correlaciones completa
print("=== Matriz de correlaciones (variables numéricas) ===")
df[columnas_numericas].corr().round(3)

In [ ]:
# Gráfico de dispersión: Age vs Attack
plt.figure(figsize=(8, 5))
plt.scatter(df["Age"], df["Attack"], color="steelblue", alpha=0.6, edgecolors="white", s=60)
plt.xlabel("Edad (Age)", fontsize=12)
plt.ylabel("Puntos de Ataque (Attack)", fontsize=12)
plt.title("Relación entre Edad y Puntos de Ataque – VNL", fontsize=13)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print(f"Correlación de Pearson r = {correlacion:.4f}")

### 2.2 Construcción del Modelo

- **Variable independiente (X):** `Age` – la edad del jugador.
- **Variable dependiente (y):** `Attack` – los puntos promedio de ataque por set.

Dividimos los datos en **80% entrenamiento** y **20% prueba**.

In [ ]:
# Definir variables
X = df[["Age"]]        # Variable independiente (debe ser 2D para sklearn)
y = df["Attack"]       # Variable dependiente

# Dividir en conjunto de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("=== División del dataset ===")
print(f"  Total de registros  : {len(df)}")
print(f"  Entrenamiento (80%) : {len(X_train)} registros")
print(f"  Prueba        (20%) : {len(X_test)} registros")

### 2.3 Entrenamiento y Predicción

Entrenamos el modelo de **regresión lineal simple** con los datos de entrenamiento y luego hacemos predicciones sobre los datos de prueba.

In [ ]:
# Crear y entrenar el modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Parámetros del modelo
print("=== Parámetros del modelo entrenado ===")
print(f"  Intercepto  (b0): {modelo.intercept_:.4f}")
print(f"  Pendiente   (b1): {modelo.coef_[0]:.4f}")
print()
print(f"  Ecuación del modelo:")
print(f"  Attack = {modelo.intercept_:.4f} + ({modelo.coef_[0]:.4f}) × Age")

In [ ]:
# Hacer predicciones sobre el conjunto de prueba
y_pred = modelo.predict(X_test)

# Mostrar comparación: valores reales vs predichos
comparacion = pd.DataFrame({
    "Age": X_test["Age"].values,
    "Attack Real": y_test.values,
    "Attack Predicho": y_pred.round(2),
    "Error": (y_test.values - y_pred).round(2)
}).reset_index(drop=True)

print("=== Comparación: Valores Reales vs Predichos (primeros 10) ===")
comparacion.head(10)

In [ ]:
# Gráfico de la recta de regresión sobre los datos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfico 1: Recta de regresión ---
ax1 = axes[0]
ax1.scatter(X_train, y_train, color="steelblue", alpha=0.6, label="Entrenamiento", s=50)
ax1.scatter(X_test, y_test, color="orange", alpha=0.8, label="Prueba (real)", s=60, zorder=5)

x_line = np.linspace(df["Age"].min(), df["Age"].max(), 100).reshape(-1, 1)
y_line = modelo.predict(x_line)
ax1.plot(x_line, y_line, color="red", linewidth=2, label="Recta de regresión")

ax1.set_xlabel("Edad (Age)", fontsize=11)
ax1.set_ylabel("Puntos de Ataque (Attack)", fontsize=11)
ax1.set_title("Regresión Lineal: Age → Attack", fontsize=12)
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

# --- Gráfico 2: Reales vs Predichos ---
ax2 = axes[1]
ax2.scatter(y_test, y_pred, color="purple", alpha=0.7, edgecolors="white", s=60)
lim_min = min(y_test.min(), y_pred.min()) - 1
lim_max = max(y_test.max(), y_pred.max()) + 1
ax2.plot([lim_min, lim_max], [lim_min, lim_max], "r--", linewidth=1.5, label="Predicción perfecta")
ax2.set_xlabel("Valores Reales", fontsize=11)
ax2.set_ylabel("Valores Predichos", fontsize=11)
ax2.set_title("Reales vs Predichos", fontsize=12)
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

### 2.4 Evaluación del Modelo

Calculamos tres métricas de error para evaluar qué tan buenas son las predicciones:

| Métrica | Significado |
|---------|-------------|
| **MAE** (Error Absoluto Medio) | Promedio del error en las mismas unidades que Attack |
| **RMSE** (Raíz del Error Cuadrático Medio) | Penaliza más los errores grandes |
| **R²** (Coeficiente de determinación) | Qué porcentaje de la variación de Attack explica el modelo (0–1) |

In [ ]:
# Cálculo de métricas
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("=== Métricas de evaluación del modelo ===")
print(f"  MAE  (Error Absoluto Medio)          : {mae:.4f}")
print(f"  MSE  (Error Cuadrático Medio)        : {mse:.4f}")
print(f"  RMSE (Raíz del Error Cuadrático)     : {rmse:.4f}")
print(f"  R²   (Coeficiente de determinación)  : {r2:.4f} ({r2*100:.1f}%)")
print()
print("📌 Interpretación:")
print(f"  El modelo se equivoca en promedio {mae:.2f} puntos de ataque (MAE).")
print(f"  El modelo explica el {r2*100:.1f}% de la variación en los puntos de ataque (R²).")

In [ ]:
# Distribución de los errores (residuos)
residuos = y_test.values - y_pred

plt.figure(figsize=(8, 4))
plt.hist(residuos, bins=12, color="steelblue", edgecolor="white", alpha=0.8)
plt.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Error = 0")
plt.xlabel("Error (Real – Predicho)", fontsize=11)
plt.ylabel("Frecuencia", fontsize=11)
plt.title("Distribución de los Errores (Residuos)", fontsize=12)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
print("Un histograma centrado en 0 indica que el modelo no tiene sesgo sistemático.")

### 2.5 Conclusión

A continuación se presenta una reflexión sobre los resultados obtenidos en este análisis.

In [ ]:
# Resumen final
print("=" * 55)
print("       RESUMEN FINAL DEL MODELO – VNL 2023")
print("=" * 55)
print(f"Variable independiente (X) : Age (Edad)")
print(f"Variable dependiente   (y) : Attack (Puntos de ataque)")
print(f"Correlación de Pearson (r) : {correlacion:.4f}")
print(f"Ecuación del modelo        : Attack = {modelo.intercept_:.2f} + {modelo.coef_[0]:.4f} × Age")
print(f"MAE                        : {mae:.4f}")
print(f"RMSE                       : {rmse:.4f}")
print(f"R²                         : {r2:.4f} ({r2*100:.1f}%)")
print("=" * 55)

#### Reflexión sobre los resultados

**Sobre la correlación:**  
La correlación de Pearson entre la edad y los puntos de ataque resultó ser débil. Esto indica que la edad por sí sola no es un buen predictor del rendimiento ofensivo de un jugador de voleibol. En la VNL, jugadores de distintas edades pueden tener niveles de ataque muy similares o muy diferentes, lo que sugiere que otros factores (posición, experiencia, equipo nacional, etc.) influyen mucho más en este indicador.

**Sobre el modelo:**  
El coeficiente R² bajo confirma que el modelo de regresión lineal simple con la edad como única variable independiente **no explica bien** la variación en los puntos de ataque. El modelo comete errores significativos en sus predicciones, como se puede ver tanto en el MAE como en el RMSE.

**Sobre su aplicabilidad:**  
A pesar de su bajo rendimiento predictivo, este ejercicio es valioso porque:  
1. Confirma que la relación entre edad y ataque en la VNL es compleja y no lineal.  
2. Sienta las bases para modelos más complejos (regresión múltiple) que incluyan variables como posición, bloqueo o saque.  
3. Demuestra que en el análisis de datos deportivos, elegir las variables correctas es tan importante como el modelo mismo.

**Recomendación:**  
Para un modelo más preciso, se sugiere explorar la relación entre **Serve** (saque) y **Attack** (ataque), o construir un modelo de **regresión múltiple** que combine varias estadísticas al mismo tiempo.